# Christenson English Prose: Parse (sentence-level)

OHCO: `chap_num, sent_num, token_num`

Source: existing `christenson_english_prose-DOC.csv`; groups chapters then applies spaCy sentence segmentation.

In [ ]:
import pandas as pd
import re
import spacy

In [ ]:
src_id = 'christenson_english_prose'
doc_path    = '../../notebooks/christenson_english_prose/christenson_english_prose-DOC.csv'
docmap_path = '../../notebooks/christenson_english_prose/christenson_english_prose-DOCMAP.csv'

## Load line-level DOC and DOCMAP, group into chapters

In [ ]:
DOC_old    = pd.read_csv(doc_path,    index_col='doc_id')
DOCMAP_old = pd.read_csv(docmap_path, index_col='doc_id')
CHAP = (DOC_old.join(DOCMAP_old).groupby('chap_num')['doc_str']
        .apply(lambda x: ' '.join(x.fillna('').astype(str))).reset_index())
print(f'{len(CHAP)} chapters')

## Apply spaCy sentence segmentation

In [ ]:
nlp = spacy.load('en_core_web_lg')
rows = []
for _, row in CHAP.iterrows():
    for s in nlp(str(row['doc_str'])).sents:
        txt = s.text.strip()
        if txt:
            rows.append((int(row['chap_num']), txt))
SENT = pd.DataFrame(rows, columns=['chap_num', 'doc_str'])
SENT['sent_num'] = SENT.groupby('chap_num').cumcount()
SENT = SENT.reset_index(drop=True); SENT.index.name = 'doc_id'
DOC = SENT[['doc_str']]; DOCMAP = SENT[['chap_num', 'sent_num']]
print(f'{len(DOC):,} sentences from {DOCMAP.chap_num.nunique()} chapters')
DOCMAP.head()

## DOC to TOKEN

In [ ]:
TOKEN = DOC.doc_str.str.split(expand=True).stack().to_frame('token_str')
TOKEN.index.names = DOC.index.names + ['token_num']
TOKEN['term_str'] = TOKEN.token_str.str.lower().str.replace(r"[^a-z']", '', regex=True)
TOKEN = TOKEN[TOKEN.term_str != ''].dropna()
TOKEN

## Save

In [ ]:
TOKEN.to_csv(f'{src_id}-TOKEN.csv')
DOC.to_csv(f'{src_id}-DOC.csv')
DOCMAP.to_csv(f'{src_id}-DOCMAP.csv')
print('Saved to notebooks/doc_tables/')